In [3]:
from datetime import datetime, timedelta
import pandas as pd
import random
from faker import Faker
fake=Faker('fr_FR')

In [4]:
import re
import unicodedata

pays_monde = [
    "France", "Canada", "Etats-Unis", "Bresil", "Argentine",
    "Maroc", "Egypte", "Nigeria", "Afrique du Sud", "Kenya",
    "Japon", "Chine", "Inde", "Coree du Sud", "Indonesie",
    "Australie", "Nouvelle-Zelande", "Mexique", "Allemagne", "Turquie"
]

villes_par_pays = {
    "France": ["Paris", "Lyon", "Marseille", "Toulouse", "Bordeaux"],
    "Canada": ["Montreal", "Toronto", "Vancouver", "Quebec", "Calgary"],
    "Etats-Unis": ["New York", "Los Angeles", "Chicago", "Houston", "Miami"],
    "Bresil": ["Sao Paulo", "Rio de Janeiro", "Brasilia", "Salvador", "Recife"],
    "Argentine": ["Buenos Aires", "Cordoba", "Rosario", "Mendoza", "La Plata"],
    "Maroc": ["Casablanca", "Rabat", "Marrakech", "Fes", "Tanger"],
    "Egypte": ["Le Caire", "Alexandrie", "Gizeh", "Louxor", "Assouan"],
    "Nigeria": ["Lagos", "Abuja", "Kano", "Ibadan", "Port Harcourt"],
    "Afrique du Sud": ["Johannesburg", "Le Cap", "Durban", "Pretoria", "Bloemfontein"],
    "Kenya": ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
    "Japon": ["Tokyo", "Osaka", "Kyoto", "Nagoya", "Sapporo"],
    "Chine": ["Pekin", "Shanghai", "Shenzhen", "Canton", "Chengdu"],
    "Inde": ["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Chennai"],
    "Coree du Sud": ["Seoul", "Busan", "Incheon", "Daegu", "Daejeon"],
    "Indonesie": ["Jakarta", "Surabaya", "Bandung", "Medan", "Yogyakarta"],
    "Australie": ["Sydney", "Melbourne", "Brisbane", "Perth", "Adelaide"],
    "Nouvelle-Zelande": ["Auckland", "Wellington", "Christchurch", "Hamilton", "Dunedin"],
    "Mexique": ["Mexico", "Guadalajara", "Monterrey", "Puebla", "Tijuana"],
    "Allemagne": ["Berlin", "Munich", "Hambourg", "Cologne", "Francfort"],
    "Turquie": ["Istanbul", "Ankara", "Izmir", "Bursa", "Antalya"]
}


def base_email_depuis_nom(nom):
    texte = unicodedata.normalize("NFKD", nom).encode("ascii", "ignore").decode("ascii").lower()
    texte = re.sub(r"[^a-z\s-]", "", texte)
    morceaux = [m for m in re.split(r"[\s-]+", texte) if m]

    if len(morceaux) >= 2:
        return f"{morceaux[0]}.{morceaux[-1]}"
    if morceaux:
        return morceaux[0]
    return "client"


def email_depuis_nom(nom, domaine="shopflow.com"):
    base = base_email_depuis_nom(nom)
    return f"{base}{random.randint(1, 9999)}@{domaine}"


def emails_invalides_adaptes(nom, domaine="shopflow.com"):
    base = base_email_depuis_nom(nom)
    return [
        f"{base}{random.randint(1, 9999)}{domaine}",
        f"{base}{random.randint(1, 9999)}@@{domaine}",
        f"{base}{random.randint(1, 9999)}@",
        f"@{domaine}",
        f"{base}{random.randint(1, 9999)}.{domaine}"
    ]


clients = []
for i in range(1, 201):
    nom = fake.name()
    pays = random.choice(pays_monde)
    city = None if i % 20 == 0 else random.choice(villes_par_pays.get(pays, [fake.city()]))

    email_valide = email_depuis_nom(nom)
    if i % 33 == 0:
        email = None
    elif i % 10 == 0:
        email = random.choice(emails_invalides_adaptes(nom))
    else:
        email = email_valide

    clients.append(
        {
            "id": i,
            "nom": nom,
            "email": email,
            "city": city,
            "pays": pays,
            "date_inscription": str(
                fake.date_between(
                    start_date=datetime.now() - timedelta(days=365),
                    end_date=datetime.now()
                )
            )
        }
    )

In [5]:
categories = ['Electronics', 'Clothing', 'Home', 'Sports', 'Books', 'Beauty']

noms_par_categorie = {
    'Electronics': ['Wireless Headphones', 'Smart Watch', 'Bluetooth Speaker', 'Laptop Stand', 'Phone Charger'],
    'Clothing': ['Basic T-Shirt', 'Slim Jeans', 'Hoodie', 'Running Shorts', 'Winter Jacket'],
    'Home': ['Table Lamp', 'Kitchen Organizer', 'Storage Box', 'Coffee Mug Set', 'Desk Mat'],
    'Sports': ['Yoga Mat', 'Dumbbells Set', 'Running Shoes', 'Water Bottle', 'Fitness Band'],
    'Books': ['Python for Beginners', 'Marketing Strategy', 'Data Analysis Handbook', 'Story Collection', 'Business Essentials'],
    'Beauty': ['Face Cream', 'Shampoo', 'Perfume', 'Lip Balm', 'Body Lotion']
}

produits = []
for i in range(1, 51):
    categorie = random.choice(categories)
    nom_produit = random.choice(noms_par_categorie[categorie])

    produits.append(
        {
            'id': i,
            'categorie': categorie,
            'name': nom_produit,
            'prix_eur':random.uniform(5.00,500.00),
            'stock':random.randint(0,500)
        }
    )

In [6]:
commandes = []
for i in range(1, 1001):
    client = random.choice(clients)
    produit = random.choice(produits)
    status = random.choices(
        ['completed', 'pending', 'cancelled', 'refunded'],
        weights=[70, 15, 10, 5],
        k=1
    )[0]
    quantite = None if i % 50 == 0 else random.randint(1, 5)

    commandes.append(
        {
            'id': i,
            'client_id': client['id'],
            'produit_id': produit['id'],
            'quantité': quantite,
            'total_eur': random.uniform(500, 2999999),
            'status': status,
            'date_commande': str(
                fake.date_between(
                    start_date=datetime(2025, 7, 1),
                    end_date=datetime(2025, 12, 31)
                )
            )
        }
    )

In [7]:
doublons = random.sample(commandes, 15)
for cmd in doublons:
    copie = cmd.copy()
    copie['id'] = len(commandes) + 1
    commandes.append(copie)

In [8]:
df_clients = pd.DataFrame(clients)
df_produits = pd.DataFrame(produits)
df_commandes = pd.DataFrame(commandes)

In [9]:
import os

os.makedirs('data/raw', exist_ok=True)


df_clients.to_csv('data/raw/client.csv', index=False)
df_produits.to_csv('data/raw/produit.csv', index=False)
df_commandes.to_csv('data/raw/commande.csv', index=False)

print('Fichiers CSV generes dans data/raw')

Fichiers CSV generes dans data/raw


In [10]:
clients = pd.read_csv('data/raw/client.csv')
clients.head()

,id,nom,email,city,pays,date_inscription
0,1,Marc Lacroix-Seguin,marc.seguin5862@shopflow.com,La Plata,Argentine,2025-08-13
1,2,Roger Perrot,roger.perrot9726@shopflow.com,Los Angeles,Etats-Unis,2026-02-27
2,3,Simone Marin du Clément,simone.clement8102@shopflow.com,Kyoto,Japon,2025-08-19
3,4,Léon Imbert,leon.imbert2512@shopflow.com,Buenos Aires,Argentine,2025-08-13
4,5,Bernard Lopez-Perrot,bernard.perrot3663@shopflow.com,Tanger,Maroc,2025-06-13
